# Part 1: Data Preprocessing & Feature Engineering 

## Data Download and Schema Overview

### Exploring Yellow Taxi Data File

In [277]:
import polars as pl
import pathlib as pathlb
from sklearn.preprocessing import StandardScaler, OneHotEncoder

file_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
base_dir = pathlb.Path('data/raw')
file_name = 'yellow_tripdata_2024-01.parquet'
file_path = base_dir/file_name

## Checks if file downloaded if not it downloads it
if not file_path.is_file() :
    base_dir.mkdir(parents=True, exist_ok=True)
    print(f" Error {file_name} not found downloading...\n")
    pl.read_parquet(file_url).write_parquet(file_path)
taxi_df = pl.read_parquet(file_path)

print(f"{"SCHEMA"}\n")
print(f"{"Name":<25}Type")
for col, type in taxi_df.schema.items() :
    print(f"{col:<25}{type}")


taxi_df.head()

SCHEMA

Name                     Type
VendorID                 Int32
tpep_pickup_datetime     Datetime(time_unit='ns', time_zone=None)
tpep_dropoff_datetime    Datetime(time_unit='ns', time_zone=None)
passenger_count          Int64
trip_distance            Float64
RatecodeID               Int64
store_and_fwd_flag       String
PULocationID             Int32
DOLocationID             Int32
payment_type             Int64
fare_amount              Float64
extra                    Float64
mta_tax                  Float64
tip_amount               Float64
tolls_amount             Float64
improvement_surcharge    Float64
total_amount             Float64
congestion_surcharge     Float64
Airport_fee              Float64


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


### Exploring Lookup Zone Data File

In [278]:
file_name ='lookup.csv'
file_path = base_dir / file_name
file_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

if not file_path.is_file():
    base_dir.mkdir(parents=True, exist_ok=True)
    print(f" Error {file_name} not found downloading...\n")
    pl.read_csv(file_url,encoding="utf8-lossy").write_csv(file_path)

lookup_df = pl.read_csv(file_path)

print(f"{"SCHEMA"}\n")
print(f"{"Name":<25}Type")
for col, type in lookup_df.schema.items() :
    print(f"{col:<25}{type}")

print(lookup_df['Borough'].value_counts())

SCHEMA

Name                     Type
LocationID               Int64
Borough                  String
Zone                     String
service_zone             String
shape: (8, 2)
┌───────────────┬───────┐
│ Borough       ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ Queens        ┆ 69    │
│ EWR           ┆ 1     │
│ Manhattan     ┆ 69    │
│ Brooklyn      ┆ 61    │
│ Unknown       ┆ 1     │
│ Staten Island ┆ 20    │
│ N/A           ┆ 1     │
│ Bronx         ┆ 43    │
└───────────────┴───────┘


## Data Cleaning


### Cleaning Taxi Parquet

In [279]:
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime',  'PULocationID', 'DOLocationID', 'fare_amount']
initial_rows = len(taxi_df)

taxi_df = taxi_df.drop_nulls(subset = cols)
null_rows = initial_rows - len(taxi_df)

taxi_df = taxi_df.filter( 
    (~pl.col("fare_amount").is_nan()) & 
    (~pl.col("trip_distance").is_nan()))
nan_rows = (initial_rows - null_rows) - len(taxi_df)


taxi_df = taxi_df.filter((pl.col('trip_distance') > 0))
distance_rows = (initial_rows - null_rows - nan_rows) - len(taxi_df)

taxi_df = taxi_df.filter((pl.col('fare_amount') > 0)& (pl.col('fare_amount') <= 500))
fare_rows = (initial_rows - null_rows - nan_rows - distance_rows) - len(taxi_df)

taxi_df = taxi_df.filter(pl.col('tpep_dropoff_datetime') > pl.col('tpep_pickup_datetime'))
time_rows = (initial_rows - null_rows - nan_rows - distance_rows - fare_rows) - len(taxi_df)
## removes incorrect dates in data
taxi_df = taxi_df.filter((pl.col('tpep_pickup_datetime').dt.year() == 2024))

year_rows = (initial_rows - null_rows - nan_rows - distance_rows - fare_rows - time_rows) - len(taxi_df)

removed_rows= initial_rows - len(taxi_df)

print("Data Cleaning Summary\n\n")
print(f"Initial row count: {initial_rows}")
print(f"Final row count: {len(taxi_df)}\n")
print(f"Total rows removed {removed_rows}")
print(f"Rows removed due to null/missing values values: {null_rows}")
print(f"Rows removed due to nan values: {nan_rows}")
print(f"Rows remove due to invalid distance: {distance_rows}")
print(f"Rows remove due to unwanted/invlaid fare range: {fare_rows}")
print(f"Rows remove due to invalid pickup/dropoff times: {time_rows}")
print(f"Rows remove due to dates being out of range : {year_rows}")

taxi_df.write_parquet('data/cleaned_data.parquet')



Data Cleaning Summary


Initial row count: 2964624
Final row count: 2869558

Total rows removed 95066
Rows removed due to null/missing values values: 0
Rows removed due to nan values: 0
Rows remove due to invalid distance: 60371
Rows remove due to unwanted/invlaid fare range: 34569
Rows remove due to invalid pickup/dropoff times: 112
Rows remove due to dates being out of range : 14


### Cleaning Lookup CSV

In [280]:
lookup_df = lookup_df.filter(
    (pl.col('Borough') != 'Unknown')&
    (pl.col('Borough') != 'N/A'))
print(lookup_df['Borough'].value_counts())

shape: (6, 2)
┌───────────────┬───────┐
│ Borough       ┆ count │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ EWR           ┆ 1     │
│ Manhattan     ┆ 69    │
│ Bronx         ┆ 43    │
│ Brooklyn      ┆ 61    │
│ Staten Island ┆ 20    │
│ Queens        ┆ 69    │
└───────────────┴───────┘


## Feature Engineering


### Joining Boroughs to Taxi Data Frame

In [281]:
taxi_df = taxi_df.join(
    lookup_df.select(["LocationID", "Borough"]),
    left_on="PULocationID",
    right_on="LocationID",
    how="left"
)
taxi_df = taxi_df.join(
    lookup_df.select(["LocationID", "Borough"]),
    left_on="DOLocationID",
    right_on="LocationID",
    how="left",
    suffix="_dropoff" 
)

In [ ]:
test_df=taxi_df
#Temporal features
taxi_df = taxi_df.with_columns( 
    #Create col that indexes pickup hour
    (pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour')),
    #Create col that indexes weekday, starting at 0
    (pl.col('tpep_pickup_datetime').dt.weekday()-1).alias('pickup_day_of_week')
    ).with_columns(
        # Creates boolean weekend col
        (pl.col('pickup_day_of_week') >= 5).alias('is_weekend')
    )
#Trip Features
taxi_df  = taxi_df.with_columns(
    (pl.col('tpep_dropoff_datetime')- pl.col('tpep_pickup_datetime'))
    #Convert trip duration minutes distance to interger
    .dt.total_minutes().alias('trip_duration_minutes')
    ).with_columns(
        #accounts for zero denominator error
        pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('trip_distance')* 60/(pl.col('trip_duration_minutes')))
        .otherwise(0) #if duration 0 -> just 0 mph
        .alias('trip_speed_mph')
    )
taxi_df = taxi_df.with_columns(
    pl.col('trip_distance').log1p().alias('log_trip_distance'))
#Fare Features
taxi_df = taxi_df.with_columns(
        pl.when(pl.col('trip_distance') > 0)
        .then(pl.col('fare_amount')/(pl.col('trip_distance')))
        .otherwise(0) #if distance 0 -> just 0
        .alias('fare_per_mile'),
        pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('fare_amount')/(pl.col('trip_duration_minutes')))
        .otherwise(0) #if distance 0 -> just 0
        .alias('fare_per_minute')
        )

In [ ]:
#Zone Features

# 1. Initialize Encoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. Select the columns exactly as they appear
# Note: Use the actual names from your df.columns if they differ
cat_features = taxi_df.select(['Borough', 'Borough_dropoff']).to_numpy()

# 3. Fit and Transform
encoded_array = encoder.fit_transform(cat_features)

# 4. Get feature names for the summary report (Requirement 3d)
encoded_cols = encoder.get_feature_names_out(['Borough', 'Borough_dropoff'])

# 5. Add back to dataframe
encoded_df = pl.from_numpy(encoded_array, schema=list(encoded_cols))

# Find columns that exist in both DataFrames to avoid DuplicateError
common_cols = set(taxi_df.columns).intersection(set(encoded_df.columns))

# Also drop the original string columns
cols_to_drop = list(common_cols) + ['Borough', 'Borough_dropoff']
# Use list(set()) to ensure we don't try to drop the same name twice
taxi_df = taxi_df.drop([c for c in set(cols_to_drop) if c in taxi_df.columns])

# Now concatenate safely
taxi_df = pl.concat([taxi_df, encoded_df], how="horizontal")

In [284]:
# Select only the columns that start with 'Borough_' (the pickups)
pickup_dist = taxi_df.select([
    pl.col("^Borough_.*$").sum()
])

print("Pickup Distribution:")
print(pickup_dist)

Pickup Distribution:
shape: (1, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Borough_B ┆ Borough_B ┆ Borough_E ┆ Borough_M ┆ … ┆ Borough_d ┆ Borough_d ┆ Borough_d ┆ Borough_ │
│ ronx      ┆ rooklyn   ┆ WR        ┆ anhattan  ┆   ┆ ropoff_Ma ┆ ropoff_Qu ┆ ropoff_St ┆ dropoff_ │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ nhattan   ┆ eens      ┆ aten      ┆ None     │
│ f64       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ Island    ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆ f64       ┆ f64       ┆ ---       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 5754.0    ┆ 22278.0   ┆ 49.0      ┆ 2.573354e ┆ … ┆ 2.584546e ┆ 133394.0  ┆ 608.0     ┆ 24898.0  │
│           ┆           ┆           ┆ 6         ┆   ┆ 6

### Target Variable Encoding

In [283]:
taxi_df = taxi_df.with_columns(
    (pl.col("tip_amount") > (pl.col("fare_amount") * 0.20))
    .cast(pl.Int8)
    .alias("high_tip")
)
print(taxi_df["high_tip"].value_counts())

shape: (2, 2)
┌──────────┬─────────┐
│ high_tip ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 0        ┆ 1094058 │
│ 1        ┆ 1775500 │
└──────────┴─────────┘


### Data Splitting and Scaling


In [286]:
from sklearn.model_selection import train_test_split


# 1. First split: 70% Train, 30% Temp (which will become Val + Test)
train_idx, temp_idx = train_test_split(
    range(len(taxi_df)),
    test_size=0.30,
    random_state=42,
    stratify=taxi_df["high_tip"].to_numpy() # Convert to numpy for sklearn compatibility
)

# 2. Second split: Split the 30% Temp into 50/50 (results in 15% Val, 15% Test)
# We use the temp_idx to grab the subset of 'high_tip' for stratification
temp_stratify = taxi_df["high_tip"][temp_idx].to_numpy()

val_idx_inner, test_idx_inner = train_test_split(
    range(len(temp_idx)),
    test_size=0.50,
    random_state=42,
    stratify=temp_stratify
)

# Map inner indices back to the original temp_idx
val_idx = [temp_idx[i] for i in val_idx_inner]
test_idx = [temp_idx[i] for i in test_idx_inner]

# 3. Create the actual Polars DataFrames using square bracket indexing
train_df = taxi_df[train_idx]
val_df = taxi_df[val_idx]
test_df = taxi_df[test_idx]

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Test size:  {len(test_df)}")

Train size: 2008690
Val size:   430434
Test size:  430434


In [288]:
from sklearn.preprocessing import StandardScaler

# Define numeric features to scale (do NOT include the 0/1 Borough columns)
# Add other numeric columns like 'trip_distance', 'fare_amount', etc.
num_features = ["trip_distance", "fare_amount", "trip_duration_minutes", "trip_speed_mph"]

scaler = StandardScaler()

# Fit ONLY on train
scaler.fit(train_df.select(num_features).to_numpy())

# Transform all three (and put back into Polars)
def scale_df(df, scaler, columns):
    scaled_values = scaler.transform(df.select(columns).to_numpy())
    scaled_df = pl.from_numpy(scaled_values, schema=columns)
    return df.drop(columns).with_columns(scaled_df)

train_df = scale_df(train_df, scaler, num_features)
val_df = scale_df(val_df, scaler, num_features)
test_df = scale_df(test_df, scaler, num_features)

In [287]:
import polars as pl

splits = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}

print(f"{'Split':<15} | {'Samples':<10} | {'High Tip %':<12}")
print("-" * 45)

for name, df in splits.items():
    count = len(df)
    # Calculate percentage of high_tip == 1
    dist = df["high_tip"].mean() * 100
    print(f"{name:<15} | {count:<10} | {dist:>10.2f}%")

Split           | Samples    | High Tip %  
---------------------------------------------
Train           | 2008690    |      61.87%
Validation      | 430434     |      61.87%
Test            | 430434     |      61.87%


In [289]:
# Identify categories
all_cols = train_df.columns
target_cols = ["tip_amount", "high_tip"]
# These were used for engineering or joining but aren't raw features for the model
excluded_cols = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", # Use hour/day instead
    "PULocationID", "DOLocationID",                 # Replaced by Borough OHE
    "store_and_fwd_flag"                            # Usually low variance/irrelevant
]

# The features actually used for training
features = [c for c in all_cols if c not in target_cols + excluded_cols]

print("--- FEATURE SUMMARY ---")
for col in features:
    dtype = train_df[col].dtype
    print(f"Feature: {col:<30} | Type: {dtype}")

print("\n--- EXCLUDED FEATURES ---")
print(f"{'Feature':<25} | {'Reason'}")
print("-" * 50)
print(f"{'PULocationID':<25} | Redundant; replaced by One-Hot Boroughs")
print(f"{'DOLocationID':<25} | Redundant; replaced by One-Hot Boroughs")
print(f"{'Pickup/Dropoff Time':<25} | Extracted into hour/day_of_week/weekend")

--- FEATURE SUMMARY ---
Feature: VendorID                       | Type: Int32
Feature: passenger_count                | Type: Int64
Feature: RatecodeID                     | Type: Int64
Feature: payment_type                   | Type: Int64
Feature: extra                          | Type: Float64
Feature: mta_tax                        | Type: Float64
Feature: tolls_amount                   | Type: Float64
Feature: improvement_surcharge          | Type: Float64
Feature: total_amount                   | Type: Float64
Feature: congestion_surcharge           | Type: Float64
Feature: Airport_fee                    | Type: Float64
Feature: pickup_hour                    | Type: Int8
Feature: pickup_day_of_week             | Type: Int8
Feature: is_weekend                     | Type: Boolean
Feature: log_trip_distance              | Type: Float64
Feature: fare_per_mile                  | Type: Float64
Feature: fare_per_minute                | Type: Float64
Feature: Borough_Bronx                